# Day 08 — Pydantic v2 + Database Connectivity

=============================================
Module 01: Python + Async + FastAPI for LLM Engineering
Vidya Sankalp | Applied GenAI Engineering

WHAT THIS FILE COVERS:
  - Pydantic BaseModel — define and validate data shapes
  - Field() — add constraints (min_length, ge, le)
  - model_validate_json() — parse and validate JSON from an LLM in one call
  - TypedDict vs BaseModel — when to use which
  - psycopg2 — connect to Postgres, run queries, load CSV data
  - Bridging prompt engineering and Python: the JSON schema in your
    system prompt BECOMES the Pydantic model in your code

REAL DATA USED:
  - customers.csv → load into Postgres + query via Pydantic model
  - products.csv  → Pydantic model for product data
  - orders.csv    → Pydantic model for order lookup

PREREQUISITES:
  - PostgreSQL running locally (or adjust DB_* settings in .env)
  - pip install psycopg2-binary pydantic[email]


In [ ]:

import csv
import json
import logging
import os
from pathlib import Path
from typing import Literal, Optional

import psycopg2                          # sync Postgres driver
import psycopg2.extras                   # provides RealDictCursor
from dotenv import load_dotenv
from pydantic import BaseModel, Field, EmailStr, field_validator, model_validator

load_dotenv()
log = logging.getLogger(__name__)
logging.basicConfig(level=logging.INFO, format="%(levelname)s [%(name)s] %(message)s")

BASE_DIR = Path(__file__).parent.parent
DATA_DIR = BASE_DIR / "data" / "datasets"




## SECTION 1: PYDANTIC v2 MODELS

═════════════════════════════════════════════════════════════════════════════
BRIDGE FROM PROMPT ENGINEERING (Module 03, Technique 04):
In your system prompt you write:
## Output Format
{"category": "URGENT|ROUTINE_MEDICAL|BILLING", "confidence": "high|medium|low"}
That JSON schema BECOMES the Pydantic model below.
model_validate_json() both parses the LLM string AND validates it.
If the LLM returns invalid JSON or wrong field values → ValidationError.


In [ ]:

class TriageOutput(BaseModel):
    """
    Pydantic model for the LLM's triage classification output.

    This model is the Python representation of the JSON schema
    defined in the ## Output Format section of the system prompt.
    Every field here corresponds to a field the LLM is told to return.

    Pydantic will:
    - Parse the JSON string (like json.loads)
    - Validate that all required fields are present
    - Validate that values match the allowed Literal options
    - Convert types if possible (e.g. "3" → 3)
    - Raise ValidationError with clear messages on failure
    """

    # Literal restricts to specific allowed values
    # If the LLM returns "urgent" (lowercase) → ValidationError
    category: Literal["URGENT", "ROUTINE_MEDICAL", "BILLING", "APPOINTMENT", "OTHER"]

    # Literal for the confidence field
    confidence: Literal["high", "medium", "low"]

    # str with min_length — empty strings are rejected
    reason: str = Field(min_length=5, description="One sentence explaining the classification")

    model_config = {"str_strip_whitespace": True}  # auto-strip whitespace from strings



In [ ]:
class CustomerRecord(BaseModel):
    """
    Pydantic model for a customer record from customers.csv / the database.

    This validates customer data on the way INTO your application.
    If CSV data is malformed (e.g. negative customer_id), Pydantic catches it.
    """

    customer_id: int = Field(gt=0, description="Must be a positive integer")
    first_name : str = Field(min_length=1, max_length=50)
    last_name  : str = Field(min_length=1, max_length=50)
    email      : EmailStr   # EmailStr validates the email format automatically
    city       : str
    state      : str = Field(min_length=2, max_length=2, description="Two-letter state code")
    country    : str = Field(default="USA")

    @property
    def full_name(self) -> str:
        """Return the customer's full name."""
        return f"{self.first_name} {self.last_name}"

    @field_validator("email")
    @classmethod
    def lowercase_email(cls, v: str) -> str:
        """Normalise email to lowercase on input."""
        return v.lower().strip()



In [ ]:
class ProductRecord(BaseModel):
    """Pydantic model for a product from products.csv."""

    product_id    : int = Field(gt=0)
    product_name  : str = Field(min_length=1)
    category      : str
    brand         : str
    price         : float = Field(gt=0, description="Price must be positive")
    stock_quantity: int   = Field(ge=0, description="Stock cannot be negative")
    description   : Optional[str] = None   # Optional means the field can be None

    @property
    def is_in_stock(self) -> bool:
        return self.stock_quantity > 0

    @property
    def price_str(self) -> str:
        return f"${self.price:.2f}"



In [ ]:
class OrderRecord(BaseModel):
    """Pydantic model for an order from orders.csv."""

    order_id      : int
    customer_id   : int
    total_amount  : float = Field(ge=0)
    payment_method: str
    order_status  : Literal["Delivered","In Transit","Processing","Pending","Cancelled","Refunded"]
    shipping_city : str
    shipping_state: str




## TypedDict: a type HINT only — Python does NOT validate at runtime

BaseModel: validates and converts at runtime (raises errors on bad data)
Use TypedDict when:
- You just want editor/IDE type hints (no validation needed)
- LangChain requires it (create_agent context_schema must be TypedDict)
- Performance matters (TypedDict has no overhead)
Use BaseModel when:
- You receive data from external sources (LLM output, CSV files, API input)
- You want validation errors to be caught early
- You are building a FastAPI endpoint (FastAPI uses Pydantic models)


In [ ]:

from typing import TypedDict



In [ ]:
class AgentContext(TypedDict, total=False):
    """
    Per-request context passed to LangChain agents.

    This MUST be TypedDict, not Pydantic BaseModel.
    LangChain 1.0 requires TypedDict for context_schema.
    total=False means all keys are optional (no KeyError risk in middleware).

    Usage:
        agent_context = {"user_id": "user_123", "session_id": "sess_456"}
        agent.astream_events(input_data, context=agent_context)
    """
    user_id   : str   # WHO is making this request (for episodic memory)
    session_id: str   # WHICH conversation session (for tracing)
    domain    : str   # WHAT domain ("ecommerce", "pharma", "general")




## SECTION 2: DATABASE CONNECTIVITY WITH psycopg2

═════════════════════════════════════════════════════════════════════════════


In [ ]:

def get_db_connection() -> psycopg2.extensions.connection:
    """
    Create a synchronous database connection from environment variables.

    Returns a psycopg2 connection object.
    Always use this inside a `with` block (or close it explicitly)
    to avoid leaking connections.

    Returns:
        An open psycopg2 connection.

    Raises:
        psycopg2.OperationalError: If the database is unreachable.
    """

    conn = psycopg2.connect(
        host    = os.environ.get("DB_HOST", "localhost"),
        port    = int(os.environ.get("DB_PORT", "5432")),
        dbname  = os.environ.get("DB_NAME", "ecommerce"),
        user    = os.environ.get("DB_USER", "postgres"),
        password= os.environ.get("DB_PASSWORD", ""),
    )

    return conn



In [ ]:
def create_tables(conn: psycopg2.extensions.connection) -> None:
    """
    Create the e-commerce tables if they don't already exist.

    Uses IF NOT EXISTS so this can be called safely multiple times.

    Args:
        conn: An open database connection.
    """

    ddl_statements = [
        """
        CREATE TABLE IF NOT EXISTS customers (
            customer_id   INTEGER PRIMARY KEY,
            first_name    VARCHAR(50)  NOT NULL,
            last_name     VARCHAR(50)  NOT NULL,
            email         VARCHAR(100) UNIQUE NOT NULL,
            phone         VARCHAR(30),
            city          VARCHAR(50),
            state         VARCHAR(2),
            zip_code      VARCHAR(10),
            country       VARCHAR(50)  DEFAULT 'USA',
            created_at    TIMESTAMP
        )
        """,
        """
        CREATE TABLE IF NOT EXISTS products (
            product_id     INTEGER PRIMARY KEY,
            product_name   VARCHAR(200) NOT NULL,
            category       VARCHAR(100),
            brand          VARCHAR(100),
            price          NUMERIC(10,2) NOT NULL,
            stock_quantity INTEGER       DEFAULT 0,
            description    TEXT,
            created_at     TIMESTAMP
        )
        """,
        """
        CREATE TABLE IF NOT EXISTS orders (
            order_id        INTEGER PRIMARY KEY,
            customer_id     INTEGER REFERENCES customers(customer_id),
            order_date      TIMESTAMP,
            total_amount    NUMERIC(10,2),
            payment_method  VARCHAR(50),
            order_status    VARCHAR(20),
            shipping_city   VARCHAR(50),
            shipping_state  VARCHAR(50),
            shipping_zip    VARCHAR(10)
        )
        """,
        """
        CREATE TABLE IF NOT EXISTS reviews (
            review_id        INTEGER PRIMARY KEY,
            product_id       INTEGER REFERENCES products(product_id),
            customer_id      INTEGER REFERENCES customers(customer_id),
            rating           INTEGER CHECK (rating BETWEEN 1 AND 5),
            review_title     VARCHAR(200),
            review_text      TEXT,
            review_date      TIMESTAMP,
            verified_purchase BOOLEAN,
            helpful_votes    INTEGER DEFAULT 0
        )
        """,
    ]

    with conn.cursor() as cur:
        for ddl in ddl_statements:
            cur.execute(ddl)
    conn.commit()
    log.info("Tables created (or already exist)")



In [ ]:
def load_csv_to_table(
    conn     : psycopg2.extensions.connection,
    csv_path : Path,
    table_name: str,
    columns  : list[str],
    limit    : int | None = None,
) -> int:
    """
    Load rows from a CSV file into a Postgres table.

    Uses ON CONFLICT DO NOTHING so re-running is safe.

    Args:
        conn       : Open database connection.
        csv_path   : Path to the CSV file.
        table_name : Target table name.
        columns    : Column names to insert (must match CSV headers).
        limit      : Optional row limit (useful for demos with large CSVs).

    Returns:
        Number of rows inserted.
    """

    rows_inserted = 0
    placeholders  = ", ".join(["%s"] * len(columns))
    col_names     = ", ".join(columns)
    sql = f"INSERT INTO {table_name} ({col_names}) VALUES ({placeholders}) ON CONFLICT DO NOTHING"

    with open(csv_path, "r", encoding="utf-8") as f:
        reader = csv.DictReader(f)

        with conn.cursor() as cur:
            for i, row in enumerate(reader):
                if limit is not None and i >= limit:
                    break

                # Build the values tuple in the same order as `columns`
                values = tuple(row.get(col, None) or None for col in columns)

                try:
                    cur.execute(sql, values)
                    rows_inserted += 1
                except Exception as e:
                    log.warning(f"Skipped row {i}: {e}")
                    conn.rollback()   # rollback the failed transaction
                    # Re-start the connection state
                    with conn.cursor() as cur2:
                        pass  # opening a new cursor resets the failed transaction
                    continue

    conn.commit()
    return rows_inserted



In [ ]:
def get_customer_by_id(conn: psycopg2.extensions.connection, customer_id: int) -> CustomerRecord | None:
    """
    Fetch a customer by ID and return a validated Pydantic model.

    This connects the database layer to the Pydantic validation layer.
    The raw database row is validated through CustomerRecord before being
    returned to the caller — any database inconsistency is caught here.

    Args:
        conn       : Open database connection.
        customer_id: The customer ID to look up.

    Returns:
        A CustomerRecord instance, or None if not found.
    """

    # RealDictCursor returns rows as dicts instead of tuples
    # This makes it easy to feed directly into Pydantic models
    with conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor) as cur:
        cur.execute(
            "SELECT * FROM customers WHERE customer_id = %s",
            (customer_id,)   # ALWAYS use parameterised queries (prevent SQL injection)
        )
        row = cur.fetchone()   # returns None if no row found

    if row is None:
        return None

    # Validate the database row through the Pydantic model
    # model_validate() works on dicts (vs model_validate_json() for strings)
    try:
        return CustomerRecord.model_validate(dict(row))
    except Exception as e:
        log.error(f"Invalid data in database for customer {customer_id}: {e}")
        return None



In [ ]:
def get_products_in_stock(
    conn      : psycopg2.extensions.connection,
    category  : str | None = None,
    limit     : int = 10,
) -> list[ProductRecord]:
    """
    Fetch in-stock products, optionally filtered by category.

    Args:
        conn    : Open database connection.
        category: Optional category filter (e.g. "Electronics").
        limit   : Max number of products to return.

    Returns:
        List of validated ProductRecord models.
    """

    with conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor) as cur:
        if category:
            cur.execute(
                """
                SELECT * FROM products
                WHERE stock_quantity > 0 AND category = %s
                ORDER BY product_name
                LIMIT %s
                """,
                (category, limit)
            )
        else:
            cur.execute(
                """
                SELECT * FROM products
                WHERE stock_quantity > 0
                ORDER BY product_name
                LIMIT %s
                """,
                (limit,)
            )

        rows = cur.fetchall()

    # Validate each row through the Pydantic model
    # Skip rows that fail validation (bad data in DB) instead of crashing
    products = []
    for row in rows:
        try:
            products.append(ProductRecord.model_validate(dict(row)))
        except Exception as e:
            log.warning(f"Skipping invalid product row: {e}")

    return products




## SECTION 3: PARSING LLM OUTPUT THROUGH PYDANTIC

═════════════════════════════════════════════════════════════════════════════


In [ ]:

def classify_and_validate(raw_llm_response: str) -> TriageOutput:
    """
    Parse and validate an LLM JSON response using Pydantic.

    model_validate_json() does THREE things in one call:
    1. Parse the JSON string (like json.loads)
    2. Validate all field types and constraints
    3. Return a typed Python object (not a raw dict)

    Args:
        raw_llm_response: Raw string from the LLM API.

    Returns:
        A validated TriageOutput instance.

    Raises:
        pydantic.ValidationError: If validation fails (logged with details).
        json.JSONDecodeError    : If the string is not valid JSON.
    """
    from pydantic import ValidationError

    # Strip markdown fences if present (LLMs often add them)
    cleaned = raw_llm_response.strip()
    if cleaned.startswith("```"):
        cleaned = cleaned.split("\n", 1)[-1].rsplit("```", 1)[0].strip()

    try:
        result = TriageOutput.model_validate_json(cleaned)
        log.info(f"Triage result: {result.category} ({result.confidence})")
        return result

    except ValidationError as e:
        # Pydantic gives field-level error details — very useful for debugging
        log.error(f"LLM response failed validation:\n{e}")
        raise




In [ ]:
if __name__ == "__main__":
    print("=" * 60)
    print("DAY 08 — Pydantic v2 + Database")
    print("=" * 60)

    # Section 1: Pydantic validation
    print("\n[1] Pydantic model_validate_json()")
    test_responses = [
        '{"category": "URGENT", "confidence": "high", "reason": "Chest pain and arm numbness described"}',
        '```json\n{"category": "BILLING", "confidence": "medium", "reason": "Customer asked about invoice"}\n```',
        '{"category": "INVALID", "confidence": "high", "reason": "x"}',   # bad category + short reason
    ]
    for raw in test_responses:
        try:
            result = classify_and_validate(raw)
            print(f"  OK: category={result.category}, confidence={result.confidence}")
        except Exception as e:
            print(f"  FAIL: {str(e)[:80]}")

    # Section 2: CustomerRecord from dict
    print("\n[2] CustomerRecord from a dict")
    customer_data = {
        "customer_id": 1001,
        "first_name": "Danielle",
        "last_name": "Johnson",
        "email": "JOHN21@EXAMPLE.NET",  # uppercase — will be normalised
        "city": "Port Matthew",
        "state": "CO",
        "country": "USA",
    }
    customer = CustomerRecord.model_validate(customer_data)
    print(f"  Name : {customer.full_name}")
    print(f"  Email: {customer.email}")   # should be lowercase

    # Section 3: Database (needs Postgres running)
    print("\n[3] Database connectivity")
    try:
        conn = get_db_connection()
        log.info("Connected to database")
        create_tables(conn)

        # Load customers
        n = load_csv_to_table(
            conn, DATA_DIR / "customers.csv",
            table_name="customers",
            columns=["customer_id","first_name","last_name","email","phone",
                     "address","city","state","zip_code","country","created_at"],
            limit=50,
        )
        print(f"  Loaded {n} customer rows")

        # Load products
        n = load_csv_to_table(
            conn, DATA_DIR / "products.csv",
            table_name="products",
            columns=["product_id","product_name","category","brand","price","stock_quantity","description","created_at"],
            limit=50,
        )
        print(f"  Loaded {n} product rows")

        # Query
        customer = get_customer_by_id(conn, 1001)
        if customer:
            print(f"\n  Customer 1001: {customer.full_name} | {customer.email}")

        products = get_products_in_stock(conn, limit=3)
        print(f"\n  In-stock products (first 3):")
        for p in products:
            print(f"    {p.product_name:30s} {p.price_str} | stock={p.stock_quantity}")

        conn.close()

    except Exception as e:
        print(f"  Database not available ({e})")
        print("  (Expected if PostgreSQL is not running — set up DB_* in .env)")


## Run the demo

Run the cell below to execute the `if __name__ == '__main__':` block.


In [ ]:
# Execute the demo for this day
import runpy
runpy.run_path('modules/day08_pydantic_database.py', run_name='__main__')
